# EOG: dynamic historical research

No financial statement values or fixed year lists are embedded in notebook code.

`FINANCIAL_SOURCE = "wrds"` reads your latest saved WRDS report. `"sec"` reads SEC standard XBRL facts, with an optional online refresh. The chosen source stays explicit because its definitions differ from the other source. The default uses your existing WRDS cache and does not prompt for credentials.

The operating data are previously reviewed observations stored separately by ticker, year, metric, units, and source. They are **not automatically refreshed**. Unavailable years stay blank, so old values can never be relabeled as a new fiscal year.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (current, *current.parents) if (p / "app/history.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the DCF-Model project folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.history import load_history, load_operating_observations, HistoryError
from app.sec import SECError

TICKER = "EOG"
HISTORY_YEARS = 5
END_YEAR = None                # None = newest annual period in the selected source
FINANCIAL_SOURCE = "wrds"      # "wrds" or "sec"; no silent source substitution
SEC_CIK = "0000821189"         # Company identifier; change with ticker for SEC
REFRESH_SEC = False            # True downloads a new snapshot; requires SEC_USER_AGENT

history = load_history(
    ticker=TICKER, years=HISTORY_YEARS, end_year=END_YEAR,
    provider=FINANCIAL_SOURCE, cik=SEC_CIK,
    refresh=REFRESH_SEC if FINANCIAL_SOURCE == "sec" else False,
    root=PROJECT_ROOT,
)
annual = history.annual
print(f"{TICKER}: {annual.index.min()} through {annual.index.max()}")
print(f"Provider: {history.provider} | retrieved UTC: {history.retrieved_at}")
print(f"Raw source: {history.raw_path.relative_to(PROJECT_ROOT)}")
for message in history.warnings:
    print("NOTE:", message)

## 1. Standardized financial history

Amounts are USD millions, except diluted weighted-average shares in millions. Names describe a common schema; the audit table records the actual provider field. EBITDA is operating income plus the selected provider's D&A. Compustat revenue, D&A, debt, and cash can have different definitions from the filed statement lines.

The old WRDS extract lacks several balance-sheet fields. Refresh it using the command at the end of this notebook to request the expanded field list. Missing fields remain blank.

In [ ]:
financial_fields = [
    "revenue", "operating_expenses", "operating_expenses_before_da",
    "ebit", "dd_a", "ebitda", "interest_expense", "pretax_income",
    "income_tax", "net_income", "operating_cash_flow", "capex",
    "cash", "receivables", "inventory", "accounts_payable",
    "debt_current", "debt_noncurrent", "total_debt", "diluted_shares",
]
history_table = annual[financial_fields].T
history_table.index.name = "Standard field"
display(history_table.round(2))

## 2. Input coverage and source audit

This shows gaps before calculations. Each retrieved value has an original field/tag, reporting period, raw value, unit scale, retrieval time, and filing reference where available. Calculated entries identify their component fields. SEC selection uses actual period dates and the latest available annual disclosure; this is not a point-in-time backtest. Company custom XBRL tags need a separate mapping and are left unavailable.

In [ ]:
coverage = pd.DataFrame({
    "Available years": annual[financial_fields].notna().sum(),
    "Requested years": len(annual),
})
coverage["Missing years"] = [
    ", ".join(map(str, annual.index[annual[field].isna()]))
    for field in financial_fields
]
display(coverage)

audit_columns = [
    "year", "metric", "value", "units", "provider", "source_field",
    "original_value", "original_units", "period_start", "period_end",
    "filed", "retrieved_at_utc", "source", "status",
]
display(history.audit.reindex(columns=audit_columns))

## 3. Financial ratios, debt and working-capital diagnostics

`total_debt` is the sum of the selected provider's current and noncurrent debt. It is not automatically a valuation debt balance. In the saved EOG extract, Compustat includes operating leases in these fields; do not add them again. SEC debt and lease tags are displayed separately for reconciliation.

The working-capital proxy is **receivables + inventory - accounts payable**. A positive annual change is an increased balance-sheet investment. Acquisition balances, joint-interest accounts, capital payables, and omitted current accounts still require review before treating this as DCF operating NWC. The loader retains one preceding year for changes; missing prior-year inputs stay blank.

In [ ]:
ratio_fields = [
    "revenue_growth_pct", "ebit_margin_pct", "ebitda_margin_pct",
    "effective_tax_rate_pct", "capex_revenue_pct",
]
display(annual[ratio_fields].T.round(2))

debt_fields = [
    "debt_current", "debt_noncurrent", "total_debt", "cash", "net_debt",
    "operating_leases_current", "operating_leases_noncurrent",
    "finance_leases_current", "finance_leases_noncurrent",
]
display(annual[debt_fields].T.round(2))

working_capital = annual[[
    "receivables", "inventory", "accounts_payable",
    "nwc_proxy", "change_nwc_proxy",
]]
display(working_capital.round(2))

## 4. Operating history aligned to the selected years

These observations were migrated from the prior notebook into a source dataset. This removes duplicated arrays and fixes their year alignment; it does not automate retrieval of new production, price, reserve, or WTI observations. A new fiscal year shows missing operating data until a reviewed source record is added. Source and definition notes remain visible below.

In [ ]:
operating, operating_audit = load_operating_observations(
    PROJECT_ROOT / "data/reference/eog_operating_history.json",
    years=annual.index, ticker=TICKER,
)
required_operating = [
    "oil_mbbl_day", "gas_mmcf_day", "ngl_mbbl_day", "total_mboe_day",
    "annual_mmboe", "realized_oil", "realized_gas", "realized_ngl",
    "loe_boe", "gpt_boe", "ga_boe", "taxes_other_than_income",
    "oil_reserves_mmbbl", "ngl_reserves_mmbbl", "gas_reserves_bcf",
    "reserves_mmboe", "wti",
]
operating = operating.reindex(columns=required_operating)
display(operating.T.round(2))
display(operating_audit)

missing_operating = operating.isna().sum(axis=1)
for year, missing in missing_operating.items():
    if missing:
        print(f"FY{year}: {missing} operating fields unavailable; new disclosures needed.")

## 5. Business-driver metrics

All calculations join on fiscal year, with no positional list matching or forward-filling. Production growth for the first displayed year is blank here because this operating table only contains displayed years. Gas is converted at EOG's disclosed 6 Mcf per Boe. EBITDA per Boe uses the selected financial provider and includes that provider's definition limitations.

In [ ]:
drivers = pd.DataFrame(index=annual.index)
total = operating["total_mboe_day"].replace(0, float("nan"))
volume = operating["annual_mmboe"].replace(0, float("nan"))
drivers["Oil mix (%)"] = 100 * operating["oil_mbbl_day"] / total
drivers["Gas mix (%)"] = 100 * (operating["gas_mmcf_day"] / 6) / total
drivers["NGL mix (%)"] = 100 * operating["ngl_mbbl_day"] / total
drivers["Annual production growth (%)"] = 100 * (volume / volume.shift() - 1)
drivers["LOE / Boe (USD)"] = operating["loe_boe"]
drivers["GP&T / Boe (USD)"] = operating["gpt_boe"]
drivers["G&A / Boe (USD)"] = operating["ga_boe"]
drivers["Capex / Boe (USD)"] = annual["capex"] / volume
drivers["Calculated EBITDA / Boe (USD)"] = annual["ebitda"] / volume
drivers["WTI less realized oil (USD/Bbl)"] = operating["wti"] - operating["realized_oil"]
display(drivers.T.round(2))

## 6. Compare the two sources when both caches exist

The comparison uses the same selected years. A missing SEC cache does not prevent exploration of WRDS data. Differences remain labeled for investigation; the notebook does not change the chosen source to make differences disappear. The debt bridge tests whether operating leases explain the observed gap.

In [ ]:
comparison_provider = "sec" if FINANCIAL_SOURCE == "wrds" else "wrds"
comparison_history = None
try:
    comparison_history = load_history(
        TICKER, HISTORY_YEARS, provider=comparison_provider, cik=SEC_CIK,
        end_year=int(annual.index.max()), refresh=False, root=PROJECT_ROOT,
    )
except (HistoryError, SECError) as error:
    print(f"Comparison unavailable: {error}")

if comparison_history is not None:
    sec_annual = annual if FINANCIAL_SOURCE == "sec" else comparison_history.annual
    wrds_annual = annual if FINANCIAL_SOURCE == "wrds" else comparison_history.annual
    comparison_rows = []
    for metric in ["revenue", "ebit", "dd_a", "capex", "operating_cash_flow"]:
        for year in annual.index:
            vendor = wrds_annual.loc[year, metric]
            filed = sec_annual.loc[year, metric]
            comparison_rows.append({
                "Year": year, "Metric": metric, "WRDS": vendor, "SEC": filed,
                "WRDS less SEC": vendor - filed,
            })
    display(pd.DataFrame(comparison_rows).round(2))
    debt_bridge = pd.DataFrame({
        "WRDS debt": wrds_annual["total_debt"],
        "SEC debt": sec_annual["total_debt"],
        "SEC operating leases": sec_annual["operating_leases"],
    })
    debt_bridge["Unexplained difference"] = (
        debt_bridge["WRDS debt"] - debt_bridge["SEC debt"]
        - debt_bridge["SEC operating leases"]
    )
    display(debt_bridge.round(2))

## 7. Historical trend assessment

The text below uses the newest selected year and its preceding year. It reports observed changes, without attributing causes that require additional evidence. Acquisition activity and commodity/product mix should be reviewed when interpreting the results.

In [ ]:
latest_year = int(annual.index.max())
previous_year = latest_year - 1

if previous_year in annual.index:
    for metric, label in [
        ("revenue", "Revenue"), ("operating_cash_flow", "Operating cash flow"),
        ("capex", "Capital expenditures"),
    ]:
        latest, previous = annual.loc[latest_year, metric], annual.loc[previous_year, metric]
        if pd.notna(latest) and pd.notna(previous) and previous != 0:
            print(f"{label}: {100 * (latest / previous - 1):+.1f}% in FY{latest_year} versus FY{previous_year}.")
        else:
            print(f"{label}: insufficient data for a year-over-year comparison.")
    print(f"Provider used: {history.provider}; inspect reconciliation before mixing sources.")
else:
    print("Select at least two years to compare trends.")

print("Next: review missing inputs, source differences, debt scope, and operating NWC before forecasting.")

## Refresh next year

The year range advances when a new annual period appears in a refreshed source. It does not advance merely because the calendar changes. To display five years and calculate the earliest change, retrieve six annual records.

**WRDS:** In a separate PowerShell window at the project root, run:

```powershell
.\.venv-wrds\Scripts\python.exe -m app.refresh_wrds --ticker EOG --years 6
```

Sign in through the WRDS prompts and approve Duo. Then rerun this notebook with `FINANCIAL_SOURCE = "wrds"`.

**SEC:** Add `SEC_USER_AGENT=DCF-Model your-contact-email` to your existing local `.env`, replacing the contact text with your email. Do not overwrite the existing file or paste credentials into the notebook. Set `FINANCIAL_SOURCE = "sec"` and `REFRESH_SEC = True`, then run the first cell. After a successful download set `REFRESH_SEC = False` for cached exploration. No SEC API key is needed. The SEC request includes your configured contact header.

SEC's [API documentation](https://www.sec.gov/search-filings/edgar-application-programming-interfaces) explains the standard concepts it exposes; custom company disclosures need a separate adapter. This notebook's SEC mapping is conservative, so unsupported tags remain blank and visible in the coverage table.

The original root-level `EOG_Financials.ipynb` remains your prior static research notebook. Continue new dynamic work here to avoid reintroducing its hardcoded arrays.